In [0]:
import os
from pyspark.sql import functions as F
from pyspark.sql.streaming import StreamingQueryException

In [0]:
# 1. Define parameter widgets (Configured for Azure defaults)
dbutils.widgets.text("catalog", "dbr_dev", "Catalog")
dbutils.widgets.text("schema", "valeriimatviiv_bronze", "Schema")
dbutils.widgets.text("volume", "earthquake_landing", "Volume / Storage Name")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
volume = dbutils.widgets.get("volume").strip()

# 2. Check if the target catalog exists in this workspace
def is_catalog_available(cat_name):
    if not cat_name:
        return False
    try:
        available_catalogs = [row[0] for row in spark.sql("SHOW CATALOGS").collect()]
        return cat_name in available_catalogs
    except Exception:
        return False

# 3. Dynamic routing
if is_catalog_available(catalog):
    # --- Azure / Paid Unity Catalog ---
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")
    base_path = f"/Volumes/{catalog}/{schema}/{volume}"
    bronze_table = f"{catalog}.{schema}.earthquakes_bronze"
else:
    # --- Databricks Free / Shared Workspace Filesystem ---
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    base_path = f"/Workspace/Shared/{volume}"
    bronze_table = f"{schema}.earthquakes_bronze"

landing_path = f"{base_path}/landing"
bronze_table_path = f"{base_path}/bronze/earthquakes_bronze"
schema_location = f"{base_path}/bronze/_schema"
checkpoint_location = f"{base_path}/bronze/_checkpoint"

In [0]:
# Ingest landing directory directly via Auto Loader (wildcard subfolder discovery)
query = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(f"{landing_path}/*")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_location)
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .start(bronze_table_path)
)

query.awaitTermination()

In [0]:
# current_user = spark.sql("SELECT current_user()").collect()[0][0]
# default_workspace_path = f"/Workspace/Users/{current_user}/delta_assignment"

# dbutils.widgets.text("base_path", default_workspace_path, "Base Workspace Path")
# base_path = dbutils.widgets.get("base_path")

# bronze_table_path = f"{base_path}/bronze/earthquakes_bronze"
# df_bronze = spark.read.format("delta").load(bronze_table_path)

# print("=== AUTOMATED BRONZE INGESTION VERIFICATION ===")
# print(f"Total Bronze Records: {df_bronze.count()} (Expected: 210)")
# print("Table Schema Columns:", df_bronze.columns)

# rescued_records = df_bronze.filter(F.col("_rescued_data").isNotNull())
# print(f"Captured Rescued Data Records: {rescued_records.count()} (Expected: 10)")